In [1]:
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, TunedThresholdClassifierCV
from sklearn.metrics import precision_score, recall_score, f1_score, make_scorer, fbeta_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold, cross_validate, RandomizedSearchCV
from joblib import dump
import numpy as np

In [2]:
data = pd.read_csv("Data/processed_data")

X = data[["annual_income_ru", "loan_ammount_ru", "int_rate_ru", "DTI", "home_ownership", "purpose"]] 
Y = data["is_loss"]
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.25, random_state=42, stratify=Y)
Y_test = pd.DataFrame(Y_test)
X_train.head()

,annual_income_ru,loan_ammount_ru,int_rate_ru,DTI,home_ownership,purpose
10024,799200.0,41440.0,0.206695,0.051852,OWN,Debt consolidation
7031,858400.0,148000.0,0.198811,0.172414,MORTGAGE,Debt consolidation
31908,651200.0,279720.0,0.145346,0.429545,MORTGAGE,other
14271,426240.0,44400.0,0.159250,0.104167,OWN,Debt consolidation
14948,695600.0,74000.0,0.143196,0.106383,RENT,Debt consolidation


In [5]:
numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, [0,1,2,3]),
        ('cat', categorical_transformer, [4,5])
    ]
)

In [ ]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', KNeighborsClassifier(weights="distance", n_jobs = -1))
])

param_dist = {
    'classifier__n_neighbors': range(3, 31),
    'classifier__p': [1, 2],

    'classifier__algorithm': ['ball_tree', 'kd_tree'],
    'classifier__leaf_size': range(20, 50),  
}

random_search = RandomizedSearchCV(estimator=pipeline, 
                            param_distributions= param_dist,
                             n_iter= 100,
                             scoring="f1",
                             cv=StratifiedKFold(random_state=42, shuffle=True),
                             random_state=42,
                             n_jobs=-1,
                             verbose=1)
random_search.fit(X_train, Y_train)
params = random_search.best_params_
print(params)

Fitting 5 folds for each of 100 candidates, totalling 500 fits


KeyboardInterrupt: 

In [7]:
best_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', TunedThresholdClassifierCV(estimator= KNeighborsClassifier(weights="distance", n_jobs = -1,
                                        p=1,n_neighbors=3,leaf_size=21,algorithm= "ball_tree"),scoring="f1", 
                                        cv=StratifiedKFold(n_splits=10,shuffle=True)))
])
best_pipeline.fit(X_train, Y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [8]:
metrics = ["precision", "recall", "roc_auc", "f1"]
cross_val_result = cross_validate(best_pipeline,
                                    X_train, Y_train,
                                   cv=StratifiedKFold(n_splits=10,shuffle=True, random_state=42),
                                   scoring=metrics,
                                   return_train_score=True,
                                   n_jobs=-1)

In [9]:
checking_metrics = ['test_roc_auc', 'train_roc_auc', "test_f1", "train_f1", "train_precision", "test_precision",
               "train_recall", "test_recall"]
def print_dict(results: dict):
    for key, value in results.items():
        print(key + ": " + f"{value}")

def get_crossval_results()->dict:
    result = {}
    for metric in checking_metrics:
        result[metric] = float(cross_val_result.get(metric).mean())
    return result
metrics_results = []

In [10]:
if len(metrics_results) == 0:
    metrics_results = [get_crossval_results()]
    print_dict(metrics_results[0])
elif len(metrics_results) == 1:
    metrics_results.append(get_crossval_results())
    for metric in checking_metrics:
        if metrics_results[1][metric] > metrics_results[0][metric]:
            print(metric + ": " + f"{metrics_results[1][metric]}" + "⬆️")
        elif metrics_results[1][metric] < metrics_results[0][metric]:
            print(metric + ": " + f"{metrics_results[1][metric]}" + "⬇️")
        else: 
            print(metric + ": " + f"{metrics_results[1][metric]}")
    
elif len(metrics_results) == 2:
    metrics_results[0] = metrics_results[1]
    metrics_results[1] = get_crossval_results()
    for metric in checking_metrics:
        if metrics_results[1][metric] > metrics_results[0][metric]:
            print(metric + ": " + f"{metrics_results[1][metric]}" + "⬆️")
        elif metrics_results[1][metric] < metrics_results[0][metric]:
            print(metric + ": " + f"{metrics_results[1][metric]}" + "⬇️")
        else: 
            print(metric + ": " + f"{metrics_results[1][metric]}")

test_roc_auc: 0.5564243675831654
train_roc_auc: 0.9999932252079018
test_f1: 0.2497061031503288
train_f1: 0.9951487737283898
train_precision: 0.9903446311453863
test_precision: 0.17619250146775364
train_recall: 1.0
test_recall: 0.42874999999999996


In [11]:
pipeline.fit(X_train, Y_train)
prediction = pipeline.predict(X_test)
recall = recall_score(Y_test,prediction)
precision = precision_score(Y_test,prediction)
print(recall)
print(precision)

0.07651912978244561
0.2
